# Snooper Project Context

This project is a Windows-only MVP for a transparent laptop-usage awareness app.

The app is intended for parent/teen accountability during productive work sessions. It is **not** secret surveillance and should avoid sensitive content capture.

The app should not record:
- keystrokes
- screenshots
- messages
- passwords
- full page contents
- detailed personal content

The main useful signal is:

> Which application/window was in the foreground, and for how long?

Foreground attention time is more important than merely knowing which processes were open.

---

# Current Product Direction

Email reporting has been dropped for now.

The current MVP direction is:

1. Start monitoring automatically when the app starts.
2. Delete old DB entries older than configured retention days.
3. Show stats/report from the last completed session on startup.
4. Run from the Windows system tray.
5. Allow pause/resume from tray.
6. When paused, stop the current monitoring session.
7. Eventually ask how long to pause; if omitted, use default pause duration.
8. Auto-resume monitoring after the pause duration.
9. Show the report/stats for the session that just ended.
10. Provide a table-style dashboard report.

DNS/domain monitoring, VPN detection, and email can be added later. They are not required for the current MVP.

---

# Current Module Structure

The current project has these Python modules:

```text
config.py
db.py
foreground_tracker.py
idle_detector.py
process_monitor.py
monitoring_controller.py
reporter.py
report_window.py
tray.py
main.py
```

For the nbdev version, notebooks should export to approximately these same modules.

---

# Current Status

The project is close to a backend/UI MVP.

Already present:

- config settings
- SQLite/fastlite DB layer
- sessions table
- foreground events table
- process events table
- foreground app/window tracking
- idle detection
- process open/close tracking
- monitoring controller
- startup housekeeping
- deletion of old data
- pause/resume support
- last/current session report data generation
- table-style report window
- tray menu
- app entry point

Recently fixed / treated as resolved:

1. Tkinter/report window threading issue
2. Empty Treeview selection crash
3. Session `status` vs `end_reason` semantic mismatch

Known pending issues, but acceptable to defer until after real Windows test:

1. Pause should show the just-ended session report automatically.
2. Custom pause blank/default behaviour should use `DEFAULT_PAUSE_MINUTES`.
3. Report app/title fallback logic may need refinement.
4. `start_session()` may need explicit `end_time=None` / `end_reason=None`.
5. Full real Windows runtime test is still required.

---

# MVP Definition

The MVP should include:

```text
System tray app
Automatic monitoring start
Start/stop/pause/resume monitoring
Session start/end saved
Foreground app/window tracking
Idle detection
Old DB data cleanup
Last session report on startup
Table-style session report window
Clean app exit
```

The MVP does **not** need:

```text
email
DNS/domain monitoring
VPN detection
browser extension
cloud sync
authentication
installer polish
```

Those can come later.

---

# Report Design

The report should be a table-style dashboard, not just plain text.

Recommended report structure:

## Summary area

Show:

- session start time
- session end time
- total duration
- active duration
- idle duration
- active percent
- idle percent
- app switch count
- distinct app count
- top app
- longest idle stretch

## Main app table

Columns:

```text
App
Total foreground time
Percent of active time
Top title
Title count
```

The table should show the top configured number of apps, controlled by:

```text
NUM_TOP_APP_IN_REPORT
```

## Per-app title detail

When the user selects an app, show that app’s top window titles.

Columns:

```text
Window title
Duration
Percent within app
```

The number of titles should be controlled by:

```text
NUM_TOP_WINDOW_IN_REPORT
```

Important report behaviour:

- Treat idle specially.
- Do not show normal window-title breakdown for idle unless useful.
- Avoid recording or displaying excessive private details.
- Long titles should eventually be trimmed or cleaned.
- Browser titles may need cleaning, but keep raw data in DB.

---

# Current Config Concepts

Important config settings include:

```text
PROCESS_MONITORING_INTERVAL_SECONDS
EXCLUDED_PROCESSES
DATABASE_PATH
FG_APP_MONITORING_INTERVAL_SECONDS
FG_APP_MONITORING_MIN_IDLE_SECONDS
STREAMING_APP_MIN_IDLE_SECONDS
DATA_RETENTION_DAYS
NUM_TOP_APP_IN_REPORT
NUM_TOP_WINDOW_IN_REPORT
SHOW_WINDOW_TITLES
DEFAULT_PAUSE_MINUTES
```

Possible future config:

```text
APP_DISPLAY_NAME_RULES
TITLE_CLEANING_RULES
DEFAULT_REPORT_ON_STARTUP
DEFAULT_REPORT_ON_PAUSE
PAUSE_OPTIONS_MINUTES
```

---

# Database Design

Current DB concepts:

## sessions

Stores monitoring sessions.

Fields conceptually:

```text
session_id
start_time
end_time
status
end_reason
```

Recommended semantics:

```text
status: open | closed
end_reason: manual_stop | manual_pause | interrupted | midnight_rollover | app_exit | unknown
```

## foreground_events

Stores foreground window intervals.

Fields:

```text
session_id
app
pid
window_title
start_time
end_time
hwnd
is_idle
id
```

This is the primary report source.

## process_events

Stores process open/close events.

Fields:

```text
session_id
pid
app
event_type
timestamp
id
```

This is supporting/debug data, not the primary productivity metric.

---

# Controller Design

`MonitorController` is the lifecycle owner.

It should own:

- running state
- current session ID
- session start/stop time
- stop event
- monitor threads
- pause timer
- rollover timer
- callbacks for UI notification/state changes

Important methods:

```text
start_monitoring()
stop_monitoring(stop_reason=...)
is_running()
run_startup_housekeeping()
generate_last_report_data()
generate_current_session_stats()
pause_for(minutes=None)
is_paused()
resume_now()
```

The tray/UI should call the controller.

The tray should not directly manage trackers or DB writes.

---

# Tray/UI Design

Use:

```text
pystray
tkinter
```

Current intended tray menu:

```text
Show last session report
Show current session stats
Pause 15 min
Pause 30 min
Pause for...
Resume now
Exit
```

Tray is for control only.

Reports should open in a separate dashboard window.

Important UI rule:

- Tkinter UI work should happen on the Tk main thread.
- Prefer `root.after(...)` for UI callbacks.
- Report windows should use `tk.Toplevel(...)` rather than creating multiple independent `Tk()` roots.

---

# Startup Flow

Recommended startup flow:

```text
main()
    create controller
    run startup housekeeping
    optionally show last completed session report
    start monitoring
    run tray loop
```

Current behaviour may be close to:

```text
create controller
start monitoring
show last session report
run tray
```

This is acceptable for MVP if it works, since the last report query should find the last closed session, not the newly opened one.

Startup housekeeping should:

1. Close stale open sessions as interrupted.
2. Delete DB entries older than `DATA_RETENTION_DAYS`.
3. Avoid deleting the active current session.

---

# Pause / Resume Behaviour

Desired final pause behaviour:

```text
User pauses monitoring
    -> stop current session with end_reason="manual_pause"
    -> generate/show report for just-ended session
    -> ask for pause duration or use default
    -> schedule auto-resume

Pause timer expires
    -> start new monitoring session
    -> show notification that monitoring resumed
```

Important design decision:

> Pause ends the current session. Resume starts a new session.

Do not try to freeze and continue the same session.

This keeps reports and DB logic simple.

---

# Foreground Tracking Design

`foreground_tracker.py` is the most important tracker.

It should repeatedly sample:

- current foreground window handle
- window title
- process ID
- process name
- idle status
- timestamp

It should emit intervals, not raw samples.

When the foreground app/window/title changes, close the previous interval and start a new one.

Idle handling:

- If user idle time exceeds configured threshold, record the interval as idle.
- For known streaming apps/domains/titles, a longer idle threshold may apply.

Foreground events are the primary report source.

---

# Idle Detection

Windows idle detection currently uses `GetLastInputInfo`.

If the user leaves a window open and walks away, time should be counted as idle rather than active use of that foreground app.

Example:

```text
Chrome open for 30 minutes
User active for 3 minutes
User idle for 27 minutes
```

Report should show:

```text
Chrome: 3 minutes
Idle: 27 minutes
```

---

# Process Monitoring

Process monitoring is useful for:

- open/close events
- debugging
- future VPN process detection

But process monitoring is secondary.

Do not use process-open duration as the main productivity metric.

---

# Reporter Design

`reporter.py` should transform raw DB rows into a plain Python report data object.

The UI should not contain business/report aggregation logic.

Preferred structure:

```text
{
    "session": {...},
    "summary": {...},
    "events": {
        "top_apps": [...],
        "titles_by_app": {...},
        "timeline": [...]
    }
}
```

Keep report generation pure and testable.

---

# Packaging / nbdev Direction

This new project will use nbdev.

Goal:

1. Copy/convert current notebooks.
2. Trim exploratory notebook content.
3. Export clean modules matching the current `.py` files.
4. Put code on GitHub.
5. Build a PyPI-installable package.
6. Eventually produce a Windows-friendly install/run path.

Suggested notebook/module mapping:

```text
00_config.ipynb              -> config.py
01_db.ipynb                  -> db.py
02_idle_detector.ipynb       -> idle_detector.py
03_foreground_tracker.ipynb  -> foreground_tracker.py
04_process_monitor.ipynb     -> process_monitor.py
05_reporter.ipynb            -> reporter.py
06_report_window.ipynb       -> report_window.py
07_monitoring_controller.ipynb -> monitoring_controller.py
08_tray.ipynb                -> tray.py
09_main.ipynb                -> main.py
```

Tests/manual checks should be added gradually.

---

# Recommended Immediate Next Steps

In the new nbdev project:

1. Create notebooks that export to the same module names.
2. Copy the current working `.py` logic into matching nbdev notebooks.
3. Keep the code as close as possible to the current working version first.
4. Do not redesign while migrating.
5. Get nbdev export working.
6. Run the app on Windows.
7. Perform the real MVP manual test.
8. Only then clean/refactor.

Real Windows MVP test:

```text
1. Start app.
2. Confirm tray appears.
3. Confirm monitoring starts.
4. Use two apps for about 30 seconds each.
5. Trigger idle briefly if practical.
6. Show current session stats.
7. Pause for 1 minute.
8. Confirm pause stops current session.
9. Confirm report appears or can be opened.
10. Confirm auto-resume starts a new session.
11. Exit cleanly.
12. Restart app.
13. Confirm last session report appears.
14. Confirm old-data cleanup does not break current data.
```

---

# Assistant Guidance For This Project

When helping with this project:

- Keep the MVP narrow.
- Prefer small, safe changes.
- Do not reintroduce email unless explicitly requested.
- Do not prioritize DNS/VPN/browser extension before the tray/report MVP works.
- Treat foreground tracking and reports as the core value.
- Be careful with Windows-specific runtime assumptions.
- Avoid large rewrites unless requested.
- When reviewing code, look first for MVP blockers:
  - import/runtime errors
  - DB schema mismatches
  - thread lifecycle bugs
  - Tkinter thread/root issues
  - session state bugs
  - report aggregation bugs
  - clean exit bugs
- Prefer table-style dashboard reports.
- Keep privacy/transparency central.

# Current Priority

Do not expand scope yet.

Current priority is to migrate the existing working project into nbdev with minimal behavioural changes, then run a real Windows MVP test.

Only after that should we improve pause/report flow, packaging, installer behaviour, DNS/VPN, or browser visibility.

I have all the code in notebook form ina different root directory, however they are not in format of nbdev. What would be the best way to migrate those notebooks under the nbs folder in this root directory in the format of nbdev?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠

I’ll verify the current nbdev workflow, then outline a minimal-change migration plan rather than redesigning the project during the move.

*[Response interrupted]*

I have all the necessary code as notebokks under a different root directory, but not in the format nbdev needs them. What would be the best way to move them under nbs folder under this root directory in the nbdev format?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I’ll check the current nbdev documentation, then give you a minimal-change migration workflow that preserves the existing notebook code.
- ⏳ `web_search(type="search", queries="['site:nbdev.fast.ai tutorial notebook #…", query="site:nbdev.fast.ai tutorial notebook #| …")` ⏳


```json {.tool}
{
  "id": "ws_07d79f67895b7bcd016a6f66b6d81881909e26a5f19c98032b",
  "name": "web_search",
  "args": {
    "type": "search",
    "queries": [
      "site:nbdev.fast.ai tutorial notebook #| default_exp #| export nbdev_export nbdev_new",
      "site:nbdev.fast.ai api notebooks migrate existing notebooks nbdev",
      "site:nbdev.fast.ai getting started nbdev config nbs_path lib_path"
    ],
    "query": "site:nbdev.fast.ai tutorial notebook #| default_exp #| export nbdev_export nbdev_new"
  },
  "result": "Server tool call executed.",
  "server": true
}
```




- ⏳ `web_search(type="search", queries="['site:nbdev.fast.ai \"nbdev-prepare\" nbd…", query="site:nbdev.fast.ai \"nbdev-prepare\" nbdev…")` ⏳


```json {.tool}
{
  "id": "ws_07d79f67895b7bcd016a6f66d152588190b780a76eccc2b998",
  "name": "web_search",
  "args": {
    "type": "search",
    "queries": [
      "site:nbdev.fast.ai \"nbdev-prepare\" nbdev3",
      "site:nbdev.fast.ai \"nbdev_test\" \"nbdev_export\" prepare",
      "site:nbdev.fast.ai \"#| export\" imports generated module"
    ],
    "query": "site:nbdev.fast.ai \"nbdev-prepare\" nbdev3"
  },
  "result": "Server tool call executed.",
  "server": true
}
```




- ⏳ `web_search(type="search", queries="['site:nbdev.fast.ai \"skip_exec: true\" n…", query="site:nbdev.fast.ai \"skip_exec: true\" not…")` ⏳


```json {.tool}
{
  "id": "ws_07d79f67895b7bcd016a6f66ea4f3c8190b1ea7806335489e3",
  "name": "web_search",
  "args": {
    "type": "search",
    "queries": [
      "site:nbdev.fast.ai \"skip_exec: true\" notebook entirely",
      "site:nbdev.fast.ai \"#| skip_exec\" notebook"
    ],
    "query": "site:nbdev.fast.ai \"skip_exec: true\" notebook entirely"
  },
  "result": "Server tool call executed.",
  "server": true
}
```





The safest approach is **copy first, then convert one notebook at a time without changing its behaviour**. Do not redesign or split code during this pass.

### 1. Confirm the destination project is configured

Current nbdev uses `pyproject.toml`. Verify that the project root contains `[project]` and `[tool.nbdev]`, with the notebook path set to `nbs` and the library path pointing to your Python package directory. If the project was created with current `nbdev-new`, this should already be correct. ([nbdev.fast.ai](https://nbdev.fast.ai/api/config.html?utm_source=openai))

Because nbdev’s command-line tools currently support Windows through **WSL rather than PowerShell[*](https://nbdev.fast.ai/api/config.html?utm_source=openai "config – nbdev")/cmd**, perform the export/build steps in Solveit’s Linux environment or WSL. The exported app itself can still be tested on native Windows. ([nbdev.fast.ai](https://nbdev.fast.ai/getting_started.html?utm_source=openai))

### 2[*](https://nbdev.fast.ai/getting_started.html?utm_source=openai "Getting Started – nbdev"). Copy—not move—the source notebooks

Keep the originals untouched until the exported package has been verified. Copy them directly as `.ipynb` files so their cells, outputs, metadata, and cell IDs are preserved.

Use the intended module order:

```text
nbs/
├── 00_config.ipynb
├── 01_db.ipynb
├── 02_idle_detector.ipynb
├── 03_foreground_tracker.ipynb
├── 04_process_monitor.ipynb
├── 05_reporter.ipynb
├── 06_report_window.ipynb
├── 07_monitoring_controller.ipynb
├── 08_tray.ipynb
└── 09_main.ipynb
```

A controlled copy script could use an explicit mapping:

```python
from pathlib import Path
from shutil import copy2

src = Path("/path/to/original/notebooks")
dst = Path("nbs")

mapping = {
    "config.ipynb": "00_config.ipynb",
    "db.ipynb": "01_db.ipynb",
    "idle_detector.ipynb": "02_idle_detector.ipynb",
    "foreground_tracker.ipynb": "03_foreground_tracker.ipynb",
    "process_monitor.ipynb": "04_process_monitor.ipynb",
    "reporter.ipynb": "05_reporter.ipynb",
    "report_window.ipynb": "06_report_window.ipynb",
    "monitoring_controller.ipynb": "07_monitoring_controller.ipynb",
    "tray.ipynb": "08_tray.ipynb",
    "main.ipynb": "09_main.ipynb",
}

for old, new in mapping.items():
    copy2(src / old, dst / new)
```

Review the mapping before running it, particularly if similarly named files already exist under `nbs`.

### 3. Add the minimum nbdev structure to each notebook

For `00_config.ipynb`, add a title and description as the first markdown cell:

```markdown
# Configuration

> Configuration settings for the Snooper application.
```

Then add a code cell near the top:

```python
#| default_exp config
```

The corresponding directives for the other notebooks are:

```python
#| default_exp db
```

```python
#| default_exp idle_detector
```

Continue with `foreground_tracker`, `process_monitor`, `reporter`, `report_window`, `monitoring_controller`, `tray`, and `main`.

`#| default_exp` selects the generated module. A code cell beginning with `#| export` is then included in that module. ([nbdev.fast.ai](https://nbdev.fast.ai/explanations/directives.html?utm_source=openai))

### [*](https://nbdev.fast.ai/explanations/directives.html?utm_source=openai "Directives – nbdev")4. Mark only production code for export

Add `#| export` to cells containing code required by the generated module—including imports, constants, functions, and classes:

```python
#| export
from pathlib import Path

DATABASE_PATH = Path.home() / ".snooper" / "snooper.db"
DATA_RETENTION_DAYS = 30
```

```python
#| export
def delete_old_data(...):
    ...
```

Use `#| exporti` for internal implementation code that should be present in the module but omitted from its public `__all__` and API documentation. ([nbdev.fast.ai](https://nbdev.fast.ai/explanations/directives.html?utm_source=openai))[*](https://nbdev.fast.ai/explanations/directives.html?utm_source=openai "Directives – nbdev")

Leave these cells unexported:

- exploratory calculations
- temporary debugging code
- demonstrations
- report previews
- manual experiments
- test invocations
- cells that start the tray or GUI merely for testing

Do **not** automatically mark every code cell with `#| export`; that would export experiments and side effects into the package.

### 5. Update cross-module imports

Once `config` has been exported, later notebooks should import from the package rather than depending on variables left in another notebook kernel:

```python
#| export
from snooper_pkg.config import (
    DATABASE_PATH,
    DATA_RETENTION_DAYS,
)
```

Replace `snooper_pkg` with the actual importable package name configured by the project.

Migrate in dependency order:

```text
config
→ db
→ idle_detector
→ foreground_tracker
→ process_monitor
→ reporter
→ report_window
→ monitoring_controller
→ tray
→ main
```

This helps expose missing imports and hidden notebook-state dependencies early.

### 6. Prevent Windows/UI experiments from running during automated tests

Ordinary unexported code cells are still treated as executable notebook tests. Put this on cells that launch Tkinter, start monitoring threads, manipulate the real DB, or require native Windows APIs:

```python
#| eval: false
```

For a notebook that cannot run at all in the Linux migration environment, temporarily add this to its title/description frontmatter:

```markdown
# Foreground tracker

> Windows foreground-window tracking.

- skip_exec: true
```

`skip_exec: true` makes `nbdev-test` skip the whole notebook; `#| eval: false` skips an individual cell. Use these narrowly rather than hiding genuine portable tests. ([nbdev.fast.ai](https://nbdev.fast.ai/explanations/directives.html?utm_source=openai))

### [*](https://nbdev.fast.ai/explanations/directives.html?utm_source=openai "Directives – nbdev")7. Export and verify one notebook at a time

After converting `00_config.ipynb`, export:

```bash
nbdev-export
```

Then inspect the generated module, expected to be approximately:

```text
snooper_pkg/config.py
```

Check that it contains:

- all required imports
- all required settings
- no exploratory code
- no unexpected initialization
- no changed defaults

Then continue with `01_db.ipynb`, export again, and repeat. `nbdev-export` generates modules from the marked cells; current nbdev also embeds notebook cell IDs in generated source so later synchronization can identify the originating cells. ([nbdev.fast.ai](https://nbdev.fast.ai/tutorials/tutorial.html?utm_source=openai))

You[*](https://nbdev.fast.ai/tutorials/tutorial.html?utm_source=openai "End-To-End Walkthrough – nbdev") can optionally add this cell at the bottom of each export notebook:

```python
#| hide
import nbdev
nbdev.nbdev_export()
```

The official tutorial recommends this as a convenient way to export after running a notebook, although the CLI command is sufficient. ([nbdev.fast.ai](https://nbdev.fast.ai/tutorials/tutorial.html?utm_source=openai))

### 8.[*](https://nbdev.fast.ai/tutorials/tutorial.html?utm_source=openai "End-To-End Walkthrough – nbdev") Test reproducibility

For each notebook:

1. Restart its kernel.
2. Run all cells from top to bottom.
3. Fix missing imports or reliance on state from another notebook.
4. Run `nbdev-export`.
5. Confirm the generated module imports successfully.

Once all notebooks migrate:

```bash
nbdev-test
```

Then:

```bash
nbdev-prepare
```

`nbdev-prepare` bundles export, notebook tests, notebook cleanup, and README generation. ([nbdev.fast.ai](https://nbdev.fast.ai/getting_started.html?utm_source=openai))

### 9. Do not use `nbdev-migrate` for[*](https://nbdev.fast.ai/getting_started.html?utm_source=openai "Getting Started – nbdev") arbitrary notebooks

`nbdev-migrate` is intended for notebooks already using older nbdev/fastpages conventions. It is not a general converter for ordinary notebooks. For your notebooks, manually adding `default_exp` and carefully selecting `export` cells is safer. ([nbdev.fast.ai](https://nbdev.fast.ai/api/migrate.html?utm_source=openai))

The recommended[*](https://nbdev.fast.ai/api/migrate.html?utm_source=openai "migrate – nbdev") migration sequence is therefore:

```text
Back up originals
→ copy notebooks into nbs
→ rename according to module order
→ add title and default_exp
→ mark production cells with export
→ update package imports
→ disable only unsafe runtime cells
→ export one module
→ compare with existing behaviour
→ proceed to the next notebook
→ run full nbdev tests
→ perform the real Windows MVP test
```

Start with `00_config.ipynb`; it is the smallest, least platform-dependent module and establishes the pattern for the remaining notebooks.